# Slice Drift Tutorial

This notebook shows how to detect drift by cohort using `SliceDriftDetector`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
import numpy as np
import pandas as pd

from drift_control.slice_drift_detector import SliceDriftDetector
from drift_control.unified_drift_detector import UnifiedDriftDetector

In [ ]:
rng = np.random.default_rng(123)

baseline = pd.DataFrame({
    'cohort': ['A'] * 400 + ['B'] * 400,
    'amount': np.concatenate([
        rng.normal(100, 15, size=400),
        rng.normal(100, 15, size=400),
    ]),
})

current = pd.DataFrame({
    'cohort': ['A'] * 400 + ['B'] * 400,
    'amount': np.concatenate([
        rng.normal(100, 15, size=400),
        rng.normal(125, 20, size=400),
    ]),
})

baseline.head()

In [ ]:
base_detector = UnifiedDriftDetector(method='ks', alpha=0.05)
slice_detector = SliceDriftDetector(detector=base_detector, by='cohort')

results = slice_detector.detect_drift(
    reference_df=baseline,
    current_df=current,
    value_column='amount',
)
SliceDriftDetector.to_frame(results)

## Interpretation

You should observe stronger drift in cohort `B` than `A`, showing why slice-level checks matter even when global averages look acceptable.